In [1]:
import bw2data as bd

In [2]:
def add_timex_ev_example_project():
    if "timex_ev_example" in bd.projects:
        bd.projects.delete_project("timex_ev_example", delete_dir=True)
        bd.projects.purge_deleted_directories()
    bd.projects.set_current("timex_ev_example")
    biosphere = bd.Database("biosphere")
    biosphere.write(
        {
            ("biosphere", "CO2"): {
                "type": "emission",
                "name": "carbon dioxide",
            },
        }
    )

    background_2020 = bd.Database("background_2020")
    background_2030 = bd.Database("background_2030")
    background_2040 = bd.Database("background_2040")

    background_2020.write({})
    background_2030.write({})
    background_2040.write({})

    background_databases = [
        background_2020,
        background_2030,
        background_2040,
    ]

    process_co2_emissions = {
        "glider": (10, 5, 2.5),  # for 2020, 2030 and 2040
        "powertrain": (20, 10, 7.5),
        "battery": (10, 5, 4),
        "electricity": (0.5, 0.25, 0.075),
        "glider_eol": (0.01, 0.0075, 0.005),
        "powertrain_eol": (0.01, 0.0075, 0.005),
        "battery_eol": (1, 0.5, 0.25),
    }

    node_co2 = biosphere.get("CO2")

    for component_name, gwis in process_co2_emissions.items():
        for database, gwi in zip(background_databases, gwis):
            database.new_node(
                component_name, name=component_name, location="somewhere"
            ).save()
            component = database.get(component_name)
            component["reference product"] = component_name
            component.save()
            production_amount = -1 if "eol" in component_name else 1
            component.new_edge(
                input=component, amount=production_amount, type="production"
            ).save()
            component.new_edge(input=node_co2, amount=gwi, type="biosphere").save()

    ELECTRICITY_CONSUMPTION = 0.2  # kWh/km
    MILEAGE = 150_000  # km
    LIFETIME = 15  # years

    # Overall mass: 1200 kg
    MASS_GLIDER = 840  # kg
    MASS_POWERTRAIN = 80  # kg
    MASS_BATTERY = 280  # kg

    if "foreground" in bd.databases:
        del bd.databases[
            "foreground"
        ]  # to make sure we create the foreground from scratch
    foreground = bd.Database("foreground")
    foreground.register()

    ev_production = foreground.new_node(
        "ev_production", name="production of an electric vehicle", unit="unit"
    )
    ev_production["reference product"] = "electric vehicle"
    ev_production.save()

    driving = foreground.new_node(
        "driving",
        name="driving an electric vehicle",
        unit="transport over an ev lifetime",
    )
    driving["reference product"] = "transport"
    driving.save()

    used_ev = foreground.new_node("used_ev", name="used electric vehicle", unit="unit")
    used_ev["reference product"] = "used electric vehicle"
    used_ev.save()

    glider_production = background_2020.get(code="glider")
    powertrain_production = background_2020.get(code="powertrain")
    battery_production = background_2020.get(code="battery")

    ev_production.new_edge(input=ev_production, amount=1, type="production").save()

    glider_to_ev = ev_production.new_edge(
        input=glider_production, amount=MASS_GLIDER, type="technosphere"
    )
    powertrain_to_ev = ev_production.new_edge(
        input=powertrain_production, amount=MASS_POWERTRAIN, type="technosphere"
    )
    battery_to_ev = ev_production.new_edge(
        input=battery_production, amount=MASS_BATTERY, type="technosphere"
    )

    glider_eol = background_2020.get(name="glider_eol")
    powertrain_eol = background_2020.get(name="powertrain_eol")
    battery_eol = background_2020.get(name="battery_eol")

    used_ev.new_edge(
        input=used_ev, amount=-1, type="production"
    ).save()  # -1 as this gets rid of a used car

    used_ev_to_glider_eol = used_ev.new_edge(
        input=glider_eol,
        amount=-MASS_GLIDER,
        type="technosphere",
    )
    used_ev_to_powertrain_eol = used_ev.new_edge(
        input=powertrain_eol,
        amount=-MASS_POWERTRAIN,
        type="technosphere",
    )
    used_ev_to_battery_eol = used_ev.new_edge(
        input=battery_eol,
        amount=-MASS_BATTERY,
        type="technosphere",
    )

    electricity_production = background_2020.get(name="electricity")

    driving.new_edge(input=driving, amount=1, type="production").save()

    driving_to_used_ev = driving.new_edge(input=used_ev, amount=-1, type="technosphere")
    ev_to_driving = driving.new_edge(input=ev_production, amount=1, type="technosphere")
    electricity_to_driving = driving.new_edge(
        input=electricity_production,
        amount=ELECTRICITY_CONSUMPTION * MILEAGE,
        type="technosphere",
    )

    from bw_temporalis import TemporalDistribution, easy_timedelta_distribution
    import numpy as np

    td_assembly_and_delivery = TemporalDistribution(
        date=np.array([-3, -2], dtype="timedelta64[M]"), amount=np.array([0.2, 0.8])
    )

    td_glider_production = TemporalDistribution(
        date=np.array([-2, -1, 0], dtype="timedelta64[Y]"),
        amount=np.array([0.7, 0.1, 0.2]),
    )

    td_produce_powertrain_and_battery = TemporalDistribution(
        date=np.array([-1], dtype="timedelta64[Y]"), amount=np.array([1])
    )

    td_use_phase = easy_timedelta_distribution(
        start=0,
        end=LIFETIME,
        resolution="Y",
        steps=(LIFETIME + 1),
        kind="uniform",  # you can also do "normal" or "triangular" distributions
    )

    td_disassemble_used_ev = TemporalDistribution(
        date=np.array([LIFETIME + 1], dtype="timedelta64[Y]"), amount=np.array([1])
    )

    td_treating_waste = TemporalDistribution(
        date=np.array([3], dtype="timedelta64[M]"), amount=np.array([1])
    )

    glider_to_ev["temporal_distribution"] = td_glider_production
    glider_to_ev.save()

    powertrain_to_ev["temporal_distribution"] = td_produce_powertrain_and_battery
    powertrain_to_ev.save()

    battery_to_ev["temporal_distribution"] = td_produce_powertrain_and_battery
    battery_to_ev.save()

    ev_to_driving["temporal_distribution"] = td_assembly_and_delivery
    ev_to_driving.save()

    electricity_to_driving["temporal_distribution"] = td_use_phase
    electricity_to_driving.save()

    driving_to_used_ev["temporal_distribution"] = td_disassemble_used_ev
    driving_to_used_ev.save()

    used_ev_to_glider_eol["temporal_distribution"] = td_treating_waste
    used_ev_to_glider_eol.save()

    used_ev_to_powertrain_eol["temporal_distribution"] = td_treating_waste
    used_ev_to_powertrain_eol.save()

    used_ev_to_battery_eol["temporal_distribution"] = td_treating_waste
    used_ev_to_battery_eol.save()

    bd.Method(("GWP", "example")).write(
        [
            (("biosphere", "CO2"), 1),
        ]
    )

In [3]:
add_timex_ev_example_project()

100%|██████████| 1/1 [00:00<00:00, 10305.42it/s]


Vacuuming database 


/Users/timodiepers/anaconda3/envs/streamlit/lib/python3.12/site-packages/bw2calc/__init__.py:45: UserWarning: 
It seems like you have an ARM architecture, but haven't installed scikit-umfpack:

    https://pypi.org/project/scikit-umfpack/

Installing it could give you much faster calculations.

  warnings.warn(UMFPACK_WARNING)


In [4]:
bd.projects.set_current("timex_ev_example")

In [5]:
bd.databases

Databases dictionary with 5 object(s):
	background_2020
	background_2030
	background_2040
	biosphere
	foreground

In [6]:
from bw_timex import TimexLCA

In [7]:
from datetime import datetime
database_date_dict = {
    "background_2020": datetime.strptime("2020", "%Y"),
    "background_2030": datetime.strptime("2030", "%Y"),
    "background_2040": datetime.strptime("2040", "%Y"),
    "foreground": "dynamic",
}

In [8]:
tlca = TimexLCA(
    demand={bd.get_node(name="driving an electric vehicle"): 1},
    method=("GWP", "example"),
    database_date_dict=database_date_dict,
    )
timeline = tlca.build_timeline()
timeline

Starting graph traversal
Calculation count: 9


/Users/timodiepers/anaconda3/envs/streamlit/lib/python3.12/site-packages/bw_timex/timex_lca.py:213: UserWarning: No edge filter function provided. Skipping all edges in background databases.
  warnings.warn(
/Users/timodiepers/anaconda3/envs/streamlit/lib/python3.12/site-packages/bw_timex/timeline_builder.py:523: Warning: Reference date 2041-01-01 00:00:00 is higher than all provided dates. Data will be taken from the closest lower year.
  warnings.warn(


,date_producer,producer_name,date_consumer,consumer_name,amount,interpolation_weights
0,2023-01-01,glider,2025-01-01,production of an electric vehicle,1176.0,"{'background_2020': 0.7, 'background_2030': 0.3}"
1,2024-01-01,glider,2025-01-01,production of an electric vehicle,168.0,"{'background_2020': 0.6, 'background_2030': 0.4}"
2,2024-01-01,powertrain,2025-01-01,production of an electric vehicle,160.0,"{'background_2020': 0.6, 'background_2030': 0.4}"
3,2024-01-01,battery,2025-01-01,production of an electric vehicle,560.0,"{'background_2020': 0.6, 'background_2030': 0.4}"
4,2025-01-01,glider,2025-01-01,production of an electric vehicle,336.0,"{'background_2020': 0.5, 'background_2030': 0.5}"
5,2025-01-01,electricity,2025-01-01,driving an electric vehicle,1875.0,"{'background_2020': 0.5, 'background_2030': 0.5}"
6,2025-01-01,production of an electric vehicle,2025-01-01,driving an electric vehicle,1.0,None
7,2025-01-01,driving an electric vehicle,2025-01-01,-1,1.0,None
8,2026-01-01,electricity,2025-01-01,driving an electric vehicle,1875.0,"{'background_2020': 0.4, 'background_2030': 0.6}"
9,2027-01-01,electricity,2025-01-01,driving an electric vehicle,1875.0,"{'background_2020': 0.3, 'background_2030': 0.7}"
